# CAD2D3D · Calco con IA propia — entrenamiento

Este cuaderno entrena el modelo que lee plantas (muros, puertas, ventanas, muebles, textos y cotas) **solo con datos propios**: plantas sintéticas generadas por CAD2D3D. No usa conjuntos de datos de terceros.

**Cómo usarlo**
1. Menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)* y *Guardar*.
2. Menú *Entorno de ejecución → Ejecutar todas*.
3. Dejá la pestaña abierta (tarda 2 a 3 horas). Al final se descarga `calco.onnx`: pasáselo a Claude.


In [ ]:
# 1 · Código de CAD2D3D (solo la carpeta del calco)
!rm -rf blackline-cad && git clone --depth 1 --filter=blob:none --sparse https://github.com/diegoalonsok-png/blackline-cad.git
%cd blackline-cad
!git sparse-checkout set ml/calco
%cd ml/calco
!ls

In [ ]:
# 2 · Dependencias
!pip -q install onnx onnxscript
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU: cambiá el tipo de entorno')

In [ ]:
# 3 · Plantas de entrenamiento (≈ 20 000, 512 px). Tarda unos 15 minutos.
!python3 dibujar_dataset.py --motor plantas_motor.jsonl --salida /content/dataset --cantidad 20000 --lado 512 --semilla 2026
!python3 dibujar_dataset.py --motor plantas_motor.jsonl --muestra /content/muestra.png --semilla 7
from IPython.display import Image as Im; Im('/content/muestra.png')

In [ ]:
# 4 · Entrenamiento (2 a 3 horas en una T4). Muestra el acierto por clase (IoU) en cada época.
!python3 entrenar.py --datos /content/dataset --epocas 40 --lado 512 --lote 24 --salida /content/calco.onnx

In [ ]:
# 5 · Descargar el modelo
from google.colab import files
files.download('/content/calco.onnx'); files.download('/content/calco.onnx.json')